# CA Reservoir with Spatial Mask

Showing the space-time tradeoff in a Cellular Automata based reservoir with a spatial mask. 

### Scientific Grounding: Dimensional Translation via Spatial Masking

The strategy implemented here—using a dense spatial mask to accelerate a slow temporal process—is a computational manifestation of **Time-Multiplexed Reservoir Computing**. 

In physical systems, when a substrate is too slow to process high-frequency temporal data, we can fold the temporal dimension into the spatial dimension. By passing a slow signal through a static, high-frequency spatial mask (a "grating"), the spatial interference creates a rapid temporal beat frequency that a slow reservoir can integrate. 

**Key Literature & Precedents:**
1. **Appeltant et al. (2011), *Nature Communications*:** "Information processing using a single dynamical node through liquid state machine." This foundational paper demonstrated that a single, slow nonlinear node (like a photonic circuit) could process complex, fast temporal signals by using a fast temporal mask to unfold the signal into a high-dimensional virtual state space.
2. **Larger et al. (2013), *Optics Express*:** Demonstrated this exact space-time folding in photonic delay reservoirs, proving that the "speed" of the computation is decoupled from the physical bandwidth of the node, bounded only by the resolution of the mask.

**Systems Synthesis:**
This mirrors biological architectures like the mammalian cochlea, which uses a spatial stiffness gradient (the basilar membrane) to translate slow, broad fluid waves into high-frequency, localized temporal firing rates in the auditory nerve. In both biology and this algorithm, "speed" is not a fundamental physical absolute, but an emergent property of spatial geometry.

**Note:** The simulation is heavily optimized, but generating the interactive animation takes a few seconds. Please wait for the progress messages below the sliders to finish before interacting.

In [1]:
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "matplotlib",
#     "numpy",
#     "pillow",
#     "ipywidgets",
#     "jupyter",
# ]
# ///

import sys
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')

from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import ipywidgets as widgets
from ipywidgets import interact

In [2]:
print(f"⏳ Please wait: Building interactive UI and computing the first frame...", flush=True)
def run_simulation(N=200, T=150, lr=0.3, sr=0.9):
    print(f"[1/3] Initializing topology for {N} cells...", flush=True)
    
    # --- 1. Setup the CA Topology (Rule 90) ---
    # OPTIMIZATION: Instead of building a dense N x N matrix and doing O(N^2) matrix multiplication,
    # we use np.roll to instantly grab the left and right neighbors in O(N) time.
    # This makes the simulation ~100x faster and uses almost zero memory.

    # --- 2. Generate the Spatial Mask (PRBS) ---
    np.random.seed(42)
    mask = (np.random.rand(N) > 0.5).astype(float)

    # --- 3. Pre-compute Space-Time States ---
    print(f"[2/3] Computing {T} time steps...", flush=True)
    raw_states = np.zeros((T, N))
    masked_states = np.zeros((T, N))

    x = np.zeros(N)
    x[N//2] = 1.0 # Initial perturbation

    for t in range(T):
        # O(N) neighbor calculation instead of O(N^2) matrix multiplication
        neighbors = np.roll(x, 1) + np.roll(x, -1)
        x_next = (1 - lr) * x + lr * np.tanh(sr * neighbors)
        x = x_next
        
        raw_states[t] = x
        masked_states[t] = x * mask

    # --- 4. Setup Animation Figure ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    fig.suptitle("Evolution of Space-Time Tradeoff", fontsize=16, color='white')

    x_axis = np.arange(N)

    line1, = ax1.plot([], [], color='#ff4d4d', lw=1.5, label='Raw CA State')
    line2, = ax2.plot([], [], color='#00ffff', lw=1.5, label='Masked CA State (Simulated Fast Signal)')

    for ax in (ax1, ax2):
        ax.set_xlim(0, N)
        ax.set_ylim(-1.1, 1.1)
        ax.set_ylabel("State Amplitude", color='white')
        ax.tick_params(colors='white')
        ax.legend(loc='upper right', facecolor='#222222', edgecolor='white', labelcolor='white')

    ax2.set_xlabel(f"Spatial Cells (N={N})", color='white')

    time_text = ax1.text(0.02, 0.95, '', transform=ax1.transAxes, fontsize=12,
                         verticalalignment='top', color='white',
                         bbox=dict(boxstyle='round', facecolor='#333333', edgecolor='white', alpha=0.8))

    # --- 5. Animation Update Function ---
    def update(frame):
        line1.set_data(x_axis, raw_states[frame])
        line2.set_data(x_axis, masked_states[frame])
        time_text.set_text(f"Time Step (Slow Clock): {frame}")
        return line1, line2, time_text

    # --- 6. Generate and Display ---
    print(f"[3/3] Generating interactive animation... (please wait)", flush=True)
    ani = FuncAnimation(fig, update, frames=T, interval=50, blit=True)
    
    ani.save("ca_evolution.gif", writer="pillow", fps=20)
    plt.close(fig)
    
    print("Done! You can now use the sliders above.", flush=True)
    return HTML(ani.to_jshtml())

# --- 7. Interactive Sliders ---
# Defaults lowered (N=200, T=150) for instant initial rendering.
# continuous_update=False prevents UI freezing while dragging.
interact(
    run_simulation,
    N=widgets.IntSlider(value=200, min=50, max=800, step=50, description='Cells (N):', continuous_update=False),
    T=widgets.IntSlider(value=150, min=50, max=400, step=50, description='Steps (T):', continuous_update=False),
    lr=widgets.FloatSlider(value=0.3, min=0.05, max=0.9, step=0.05, description='Leak (lr):', continuous_update=False),
    sr=widgets.FloatSlider(value=0.9, min=0.1, max=1.5, step=0.1, description='Spec. Rad (sr):', continuous_update=False)
);

⏳ Please wait: Building interactive UI and computing the first frame...


interactive(children=(IntSlider(value=200, continuous_update=False, description='Cells (N):', max=800, min=50,…